# 02 · Baselines: NDVI threshold (B1) and random forest (B2), site cross-validation
The two simple models the U-Net and DINOv3 have to beat. Both are scored with leave-one-site-out cross-validation on the Train/Val sites only; the locked test set is not touched.

## Environment
Runs in the `mangrove` conda env (`2_Code\environment.yml`; setup in `RUN_LOCALLY.md`).

## Paths
`paths.py` defines every folder in one place; Jupyter is launched from the SDP folder, so no paths need editing.
* `3_Data/`: small inputs (sites, labels, image list, folds)
* `6_Work/`: everything the pipeline rebuilds (images, chips, datasets), about 4 GB
* `4_Results/`: tables and figures

In [ ]:
import os, subprocess, sys
from pathlib import Path

for _d in [Path.cwd(), *Path.cwd().parents]:          # works from SDP, SDP/2_Code or SDP/2_Code/notebooks
    if (_d / 'paths.py').exists():
        CODE = _d; break
    if (_d / '2_Code' / 'paths.py').exists():
        CODE = _d / '2_Code'; break
else:
    raise FileNotFoundError('2_Code/paths.py not found: start Jupyter inside the SDP folder')
sys.path.insert(0, str(CODE))
import paths as P

PROJECT_ROOT, LAYOUT = P.PROJECT_ROOT, 'sdp'
WORK, SITES, LABELS, SCENES, FOLDS, QA_CSV = P.WORK, P.SITES, P.LABELS, P.SCENES, P.FOLDS, P.QA_CSV
DATASETS, RESULTS = P.DATASETS, P.RESULTS
TRAINVAL = P.TRAIN_VAL            # the Test_Dataset folder is never opened in these notebooks

def run_step(*args):
    """Run 2_Code/run_all.py with these arguments and show its output here, live."""
    env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([sys.executable, str(CODE / 'run_all.py'), *map(str, args)], cwd=PROJECT_ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('run_all.py failed: see the messages above')

print(P.describe())

In [ ]:
import json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 11})
pd.set_option('display.width', 160)

In [ ]:
from chipio import load_chip, get_label, chip_path

def stretch(a, hi):
    return np.clip(np.nan_to_num(a) / hi, 0, 1)

def rgb(chip):        # true colour: B04, B03, B02
    im = chip['image']; return np.dstack([stretch(im[2], .25), stretch(im[1], .25), stretch(im[0], .25)])

def false_colour(chip):   # B08, B04, B03: healthy vegetation looks red
    im = chip['image']; return np.dstack([stretch(im[3], .45), stretch(im[2], .25), stretch(im[1], .25)])

def label_img(lab):   # 0 = sand colour, 1 = green, 255 (no label) = grey
    out = np.zeros(lab.shape + (3,)); out[lab == 0] = (.93, .91, .84); out[lab == 1] = (0, .55, .24); out[lab == 255] = (.6, .6, .6)
    return out

## 1 · Folds
The 6 Abu Dhabi sites form 5 folds (Al Aryam and Al Nouf are 1.5 km apart and count as one). Each model is trained 5 times: one fold is held out, the model learns from the other four and is scored on the held-out site. The final score is the mean of the 5 rounds. Splitting by whole site keeps held-out chips away from training chips, avoiding the leakage seen in the earlier SAM work.

In [ ]:
folds = pd.read_csv(FOLDS); cvf = folds[folds.role == 'cv'].copy(); cvf['fold'] = cvf.fold.astype(int)
display(cvf.groupby('fold').site.apply(', '.join).rename('site(s)').to_frame())
n = cvf.fold.nunique()
grid = np.ones((n, n)); np.fill_diagonal(grid, 0)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.imshow(grid, cmap=plt.matplotlib.colors.ListedColormap(['#E8A33D', '#2E5597']))
for i in range(n):
    for j in range(n):
        ax.text(j, i, 'score' if i == j else 'train', ha='center', va='center', color='white', fontsize=9)
ax.set_xticks(range(n), [f'fold {k}' for k in range(1, n + 1)]); ax.set_yticks(range(n), [f'round {k}' for k in range(1, n + 1)])
ax.set_title('Leave-one-site-out: each round holds one fold back'); plt.tight_layout(); plt.show()

## 2 · The two baselines
* **B1, NDVI threshold:** a pixel is mangrove if NDVI > *t* and MNDWI < *w* (not water). Both cut-offs are learned from the 4 training folds.
* **B2, random forest:** about 200 decision trees vote on each pixel from its 8 values (6 bands, NDVI, MNDWI). It sees one pixel at a time, without shape or context. Settings and decision threshold are tuned on an inner fold, never on the held-out one.

Each runs twice: **(a)** with all 2020 images, **(b)** with hazy images (dark-water red > 0.05) dropped from training.

`RUN_CV = True` reruns the full cross-validation; with `False` the saved results are loaded.

In [ ]:
RUN_CV = True
BASE = P.BASELINES                                   # 4_Results/Baselines: tables and figures
PREDS = P.WORK / 'runs' / 'Baselines' / 'preds'      # 6_Work/runs/Baselines/preds: predicted masks
if RUN_CV:
    run_step('--step', 'baselines')                  # = python 2_Code/run_all.py --step baselines
else:
    print('RUN_CV = False: loading saved results from', BASE)

## 3 · Results per fold
Uses the official scores (`cv_results.csv`, written by `score.py` with the shared `evaluate.py`) when they exist. Otherwise the notebook scores the saved predictions itself and marks the numbers provisional.

In [ ]:
from evaluate import confusion, metrics_from_counts
cv_path = BASE / 'cv_results.csv'
cvcat_path = BASE / 'cv_catalog_2020.csv'
MODELS = [('B1', 'a_all_dates'), ('B2', 'a_all_dates'), ('B1', 'b_no_haze'), ('B2', 'b_no_haze')]
cv, SOURCE = None, None
if cv_path.exists():
    cv = pd.read_csv(cv_path); SOURCE = 'official (score.py / evaluate.py)'
elif cvcat_path.exists() and PREDS.exists():
    cvcat = pd.read_csv(cvcat_path); rows = []
    for m, v in MODELS:
        pdir = PREDS / f'{m}_{v}'
        for k in sorted(cvcat.fold.unique()):
            held = cvcat[cvcat.fold == k]; tp = fp = fn = 0; n = 0
            for _, r in tqdm(held.iterrows(), total=len(held), desc=f'{m} {v} fold {k}', leave=False):
                p = pdir / f'{r.chip_id}.npz'
                if not p.exists():
                    continue
                chip = load_chip(chip_path(TRAINVAL, r)); truth, labelled = get_label(chip, 'v4019')
                pred = np.load(p)['mask'].astype(bool)
                a, b, c, _ = confusion(pred, truth, chip['valid'] & labelled); tp += a; fp += b; fn += c; n += 1
            if n == len(held):
                rows.append({'model': m, 'variant': v, 'fold': k, 'held_site': ','.join(sorted(held.site.unique())), 'chips': n,
                             **{x: float(y) for x, y in metrics_from_counts(tp, fp, fn).items()}})
    cv = pd.DataFrame(rows) if rows else None
    SOURCE = 'PROVISIONAL (scored here from saved predictions; folds still running are left out)'
if cv is None or cv.empty:
    print('RESULTS PENDING: the baseline run has not written predictions yet. Rerun this notebook when it finishes.')
else:
    print('Source:', SOURCE)
    display(cv[['model', 'variant', 'fold', 'held_site', 'chips', 'iou', 'f1', 'precision', 'recall']].round(3))

In [ ]:
if cv is not None and not cv.empty:
    folds_sorted = sorted(cv.fold.unique()); w = .2
    colors = {('B1', 'a_all_dates'): '#9DB7DA', ('B2', 'a_all_dates'): '#2E5597',
              ('B1', 'b_no_haze'): '#F2C98A', ('B2', 'b_no_haze'): '#C27A12'}
    fig, ax = plt.subplots(figsize=(12, 4.8))
    for i, key in enumerate(MODELS):
        d = cv[(cv.model == key[0]) & (cv.variant == key[1])].set_index('fold').reindex(folds_sorted)
        ax.bar(np.arange(len(folds_sorted)) + (i - 1.5) * w, d.iou, w, color=colors[key], label=f'{key[0]} ({key[1]})')
    sites_lbl = cv.drop_duplicates('fold').set_index('fold').reindex(folds_sorted).held_site.str.replace(',', ' +\n')
    ax.set_xticks(range(len(folds_sorted)), [f'fold {k}\n{s}' for k, s in zip(folds_sorted, sites_lbl)], fontsize=8)
    ax.set_ylabel('IoU on the held-out site'); ax.set_ylim(0, 1); ax.legend(ncol=4, fontsize=8)
    ax.set_title('Baselines per fold: score on the site each model never saw'); plt.tight_layout(); plt.show()

    summary = (cv.groupby(['model', 'variant'])[['iou', 'f1', 'precision', 'recall']]
                 .agg(['mean', 'std']).round(3))
    summary.columns = [f'{a} {b}' for a, b in summary.columns]
    print('Mean and standard deviation across folds:'); display(summary)
    best = cv.groupby(['model', 'variant']).iou.mean().idxmax()
    print(f"Bar to beat for the U-Net: {best[0]} ({best[1]}) with mean IoU {cv.groupby(['model', 'variant']).iou.mean().max():.3f}")

## 4 · Random forest feature importance
Importance of each band, averaged over the 5 folds. High SWIR (B11, B12) importance would support the exploration result that SWIR separates mangrove from other plants.

In [ ]:
imp_path = BASE / 'rf_feature_importances.csv'
if imp_path.exists():
    imp = pd.read_csv(imp_path); feats = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'NDVI', 'MNDWI']
    mean = imp.groupby('variant')[feats].mean().T
    ax = mean.plot.bar(figsize=(10, 4), color=['#2E5597', '#C27A12'])
    ax.set_ylabel('mean importance'); ax.set_title('Random forest feature importance (mean over folds)'); plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()
else:
    print('RESULTS PENDING: rf_feature_importances.csv not written yet.')

## 5 · Example predictions
Three held-out chips: true colour, 10 m label, B1 prediction and B2 prediction (all-dates variant). Green = predicted mangrove.

In [ ]:
pa, pb = PREDS / 'B1_a_all_dates', PREDS / 'B2_a_all_dates'
if cvcat_path.exists() and pa.exists() and pb.exists():
    cvcat = pd.read_csv(cvcat_path)
    have = cvcat[cvcat.chip_id.map(lambda c: (pa / f'{c}.npz').exists() and (pb / f'{c}.npz').exists())]
    picks = have.sort_values('mangrove_pct_v4019', ascending=False).drop_duplicates('site').head(3)
    fig, axes = plt.subplots(len(picks), 4, figsize=(12, 3.1 * len(picks)))
    axes = np.atleast_2d(axes)
    for i, (_, r) in enumerate(picks.iterrows()):
        chip = load_chip(chip_path(TRAINVAL, r))
        b1 = np.load(pa / f'{r.chip_id}.npz')['mask']; b2 = np.load(pb / f'{r.chip_id}.npz')['mask']
        for j, (img, t) in enumerate([(rgb(chip), 'true colour'), (label_img(chip['label_v4019']), '10 m label'),
                                      (label_img(b1.astype(np.uint8)), 'B1 NDVI threshold'), (label_img(b2.astype(np.uint8)), 'B2 random forest')]):
            axes[i, j].imshow(img); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
            if i == 0: axes[i, j].set_title(t)
            if j == 0: axes[i, j].set_ylabel(f'{r.site}\nfold {int(r.fold)}', fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print('RESULTS PENDING: predictions not available yet.')

## 6 · Written summary
`RESULTS.md` from the baseline run, if present.

In [ ]:
from IPython.display import Markdown, display as _d
res_md = BASE / 'RESULTS.md'
if res_md.exists():
    _d(Markdown(res_md.read_text(encoding='utf-8')))
else:
    print('RESULTS PENDING: RESULTS.md has not been written yet. The numbers above, if shown, come from the saved predictions.')